In [31]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import os

def plot(path):
    df_results = pd.read_csv(path)

    successful_results = df_results[df_results['success']].copy()

    pivot_mmd = successful_results.pivot_table(values='mmd_score', 
                                            index='num_bins', 
                                            columns='g', 
                                            aggfunc='mean')

    pivot_time = successful_results.pivot_table(values='execution_time_sec', 
                                            index='num_bins', 
                                            columns='g', 
                                            aggfunc='mean')

    X_mmd = pivot_mmd.columns.values
    Y_mmd = pivot_mmd.index.values
    X_mmd, Y_mmd = np.meshgrid(X_mmd, Y_mmd)
    Z_mmd = pivot_mmd.values

    X_time = pivot_time.columns.values
    Y_time = pivot_time.index.values
    X_time, Y_time = np.meshgrid(X_time, Y_time)
    Z_time = pivot_time.values

    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'surface'}, {'type': 'surface'}]],
        subplot_titles=('MMD Score vs g vs num_bins', 'Execution Time vs g vs num_bins')
    )

    fig.add_trace(
        go.Surface(z=Z_mmd, x=pivot_mmd.columns, y=pivot_mmd.index, 
                colorscale='Viridis', colorbar=dict(x=0.45, y=0.5)),
        row=1, col=1
    )

    fig.add_trace(
        go.Surface(z=Z_time, x=pivot_time.columns, y=pivot_time.index, 
                colorscale='Plasma', colorbar=dict(x=1.0, y=0.5)),
        row=1, col=2
    )

    min_mmd_idx = np.unravel_index(np.nanargmin(Z_mmd), Z_mmd.shape)
    min_g = pivot_mmd.columns[min_mmd_idx[1]]
    min_bins = pivot_mmd.index[min_mmd_idx[0]]
    min_mmd = Z_mmd[min_mmd_idx]

    min_time_idx = np.unravel_index(np.nanargmin(Z_time), Z_time.shape)
    min_time_g = pivot_time.columns[min_time_idx[1]]
    min_time_bins = pivot_time.index[min_time_idx[0]]
    min_time = Z_time[min_time_idx]

    fig.update_layout(
        title_text='Comparison: MMD Score vs Execution Time',
        scene=dict(
            xaxis_title='Parameter g',
            yaxis_title='Number of Bins',
            zaxis_title='MMD Score',
            yaxis_type='log'
        ),
        scene2=dict(
            xaxis_title='Parameter g',
            yaxis_title='Number of Bins',
            zaxis_title='Time (seconds)',
            yaxis_type='log'
        ),
        width=1400,
        height=700
    )

    fig.add_trace(
        go.Scatter3d(
            x=[min_g], y=[min_bins], z=[min_mmd],
            mode='markers', marker=dict(size=6, color='red'),
            name=f'Best MMD: {min_mmd:.4f}',
            showlegend=True
        ),
        row=1, col=1
    )

    fig.add_annotation(
        x=0.25, y=0.05, xref="paper", yref="paper",
        text=f"Best MMD: g={min_g}, bins={min_bins}, MMD={min_mmd:.6f}",
        showarrow=False,
        font=dict(color="red")
    )

    fig.add_annotation(
        x=0.75, y=0.05, xref="paper", yref="paper",
        text=f"Best Time: g={min_time_g}, bins={min_time_bins}, Time={min_time:.2f}s",
        showarrow=False,
        font=dict(color="green")
    )

    fig.show()
    return min_g, min_bins

In [32]:
import os
import pandas as pd

folders = [
    "results/adult",
    "results/blobs",
    "results/german",
    "results/heloc",
    "results/compas",
    "results/heart",
    "results/gaussian",
]

all_files = []
for folder in folders:
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.endswith(".csv"):
                all_files.append(os.path.join(root, file))

# Read all CSVs and concatenate into a single DataFrame
all_data = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)

all_data

,g,num_bins,execution_time_sec,compressed_size,mmd_score,success
0,0,1,0.000280,16,5.791128e-03,True
1,1,1,0.000443,32,7.008183e-03,True
2,2,1,0.002456,64,5.688428e-03,True
3,3,1,0.001779,128,2.182973e-03,True
4,4,1,0.000078,256,1.114062e-03,True
...,...,...,...,...,...,...
6715,3,256,0.001148,16384,5.811709e-07,True
6716,4,256,0.001786,16384,5.811709e-07,True
6717,5,256,0.001187,16384,5.811709e-07,True
6718,6,256,0.001181,16384,5.811709e-07,True


In [47]:
grouped = all_data.groupby(['num_bins', 'g'], as_index=False)['mmd_score'].sum()
grouped_sorted = grouped.sort_values('mmd_score', ascending=True)

grouped_sorted

,num_bins,g,mmd_score
28,32,4,0.017066
29,32,5,0.017066
30,32,6,0.017066
47,128,7,0.017066
46,128,6,0.017066
45,128,5,0.017066
44,128,4,0.017066
43,128,3,0.017066
31,32,7,0.017066
27,32,3,0.020094


In [51]:
plot("results/blobs/blobs_10000_1000_gaussian.csv")

(4, 32)